# Contextual Chunk Headers (CCH)

## Overview

Individual chunks often lack context. A chunk might say *"the company reduced emissions by 30%"* but never mention which company. CCH solves this by **prepending a header** (like the document title) to each chunk before embedding.

| Without CCH | With CCH |
|---|---|
| `"the company reduced emissions by 30%"` | `"Document Title: Nike 2023 Annual Report\n\nthe company reduced emissions by 30%"` |
| Query "Nike climate change" → low match | Query "Nike climate change" → **high match** |

## Why It Works

- Chunks often refer to subjects via pronouns or implicit references
- Without context, they can't be retrieved for the right queries
- Adding the document title is the **simplest and most impactful** form of CCH

## Models Used

- **LLM**: `gemma3:4b` via Ollama — generates the document title
- **Reranker**: `cross-encoder/ms-marco-MiniLM-L-6-v2` — measures relevance scores
- **Tokenizer**: `google/gemma-3-12b-it` — for truncating long documents

![Contextual Chunk Headers](./images/contextual_chunk_headers.svg)

---
## Step 0: Import Packages

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ollama import chat
from transformers import AutoTokenizer
from sentence_transformers import CrossEncoder

---
## Step 1: Download Data (if needed)

In [2]:
import os
os.makedirs("data", exist_ok=True)

if not os.path.exists("data/nike_2023_annual_report.txt"):
    !wget -O data/nike_2023_annual_report.txt https://raw.githubusercontent.com/NirDiamant/RAG_TECHNIQUES/main/data/nike_2023_annual_report.txt
    print("Downloaded Nike annual report")
else:
    print("Data file already exists")

Data file already exists


---
## Step 2: Load the Document and Split into Chunks

In [3]:
FILE_PATH = "data/nike_2023_annual_report.txt"

with open(FILE_PATH, "r") as f:
    document_text = f.read()

print(f"Document length: {len(document_text)} characters")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800, chunk_overlap=0, length_function=len
)
documents = text_splitter.create_documents([document_text])
chunks = [doc.page_content for doc in documents]

print(f"Split into {len(chunks)} chunks")
print(f"\nSample chunk (chunk 0):\n{chunks[0][:200]}...")

Document length: 374938 characters
Split into 500 chunks

Sample chunk (chunk 0):
FORM 10-K FORM 10-KUNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C.
Washington, D.C. 20549
FORM 10-K 
(Mark One)
☑ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(D) OF THE SECURITIES EXCH...


---
## Step 3: Generate a Descriptive Document Title

We ask the LLM to read the beginning of the document and produce a descriptive title. This title will become the **chunk header** prepended to every chunk.

If you already have good document titles (e.g., from filenames or metadata), you can skip this step and use those directly.

In [4]:
MAX_CONTENT_TOKENS = 4000
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-12b-it")

# Truncate document to fit in context window
tokens = tokenizer.encode(document_text, disallowed_special=())
truncated_tokens = tokens[:MAX_CONTENT_TOKENS]
truncated_text = tokenizer.decode(truncated_tokens)

num_tokens = min(len(tokens), MAX_CONTENT_TOKENS)
print(f"Document: {len(tokens)} tokens, truncated to {num_tokens} tokens for title generation")

# Build the title generation prompt
truncation_note = ""
if num_tokens >= MAX_CONTENT_TOKENS:
    truncation_note = (
        f"Also note that the document text provided below is just the first ~3000 words "
        f"of the document. That should be plenty for this task. Your response should still "
        f"pertain to the entire document, not just the text provided below."
    )

prompt = f"""INSTRUCTIONS
What is the title of the following document?

Your response MUST be the title of the document, and nothing else. DO NOT respond with anything else.

{truncation_note}

DOCUMENT
{truncated_text}
""".strip()

# Call the LLM
response = chat(
    model="gemma3:4b",
    messages=[{"role": "user", "content": prompt}],
    options={"num_predict": MAX_CONTENT_TOKENS, "temperature": 0.2},
)
document_title = response.message.content.strip()

print(f"\nGenerated document title: {document_title}")

Document: 90383 tokens, truncated to 4000 tokens for title generation

Generated document title: NIKE, INC.


---
## Step 4: See the Impact — Compare With and Without Header

Let's pick a specific chunk and measure its relevance to a query **with and without** the document title header.

We use a **CrossEncoder reranker** to score each (query, chunk) pair. Higher score = more relevant.

In [5]:
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Reranker loaded")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reranker loaded


In [6]:
CHUNK_INDEX = 86
QUERY = "Nike climate change impact"

chunk_text = chunks[CHUNK_INDEX]
chunk_without_header = chunk_text
chunk_with_header = f"Document Title: {document_title}\n\n{chunk_text}"

# Score both versions
scores = reranker.predict([
    (QUERY, chunk_without_header),
    (QUERY, chunk_with_header),
])

print(f"Chunk header: Document Title: {document_title}")
print(f"\nChunk text:\n{chunk_text}")
print(f"\nQuery: {QUERY}")
print(f"\nSimilarity WITHOUT header: {scores[0]:.4f}")
print(f"Similarity WITH header:    {scores[1]:.4f}")

Chunk header: Document Title: NIKE, INC.

Chunk text:
Given the broad and global scope of our operations, we are particularly vulnerable to the physical risks of climate change, such 
as shifts in weather patterns. Extreme weather conditions in the areas in which our retail stores, suppliers, manufacturers, 
customers, distribution centers, offices, headquarters and vendors are located could adversely affect our operating results and 
financial condition. Moreover, natural disasters such as earthquakes, hurricanes, wildfires, tsunamis, floods or droughts, whether 
occurring in the United States or abroad, and their related consequences and effects, including energy shortages and public 
health issues, have in the past temporarily disrupted, and could in the future disrupt, our operations, the operations of our

Query: Nike climate change impact

Similarity WITHOUT header: -4.2102
Similarity WITH header:    6.4446


This chunk is clearly about the impact of climate change on some organization, but it doesn't explicitly say "Nike." So the relevance to "Nike climate change impact" is low without the header. By simply adding the document title, the similarity score jumps dramatically.

---
## Step 5: Try Another Chunk

In [7]:
CHUNK_INDEX_2 = 90

chunk_text_2 = chunks[CHUNK_INDEX_2]
chunk_without_header_2 = chunk_text_2
chunk_with_header_2 = f"Document Title: {document_title}\n\n{chunk_text_2}"

scores_2 = reranker.predict([
    (QUERY, chunk_without_header_2),
    (QUERY, chunk_with_header_2),
])

print(f"Chunk text:\n{chunk_text_2}")
print(f"\nQuery: {QUERY}")
print(f"\nSimilarity WITHOUT header: {scores_2[0]:.4f}")
print(f"Similarity WITH header:    {scores_2[1]:.4f}")

Chunk text:
we could be late in delivering, or be unable to deliver, products to our customers. These events could result in reputational 
damage, lost sales, cancellation charges or markdowns, all of which could have an adverse ef fect on our business, results of 
operations and financial condition.
Our financial condition and results of operations have been, and could in the future be, adversely affected by a 
pandemic, epidemic or other public health emergency.
Pandemics, including the COVID-19 pandemic, and other public health emergencies, and preventative measures taken to 
contain or mitigate such crises have caused, and may in the future cause, business slowdown or shutdown in af fected areas and

Query: Nike climate change impact

Similarity WITHOUT header: -11.0835
Similarity WITH header:    -1.1869


---
## Evaluation Results (from the CCH paper)

CCH was evaluated on the KITE benchmark (Knowledge-Intensive Task Evaluation) across 4 datasets and 50 questions:

| Dataset | Without CCH | With CCH |
|---|---|---|
| AI Papers | 4.5 | 4.7 |
| BVP Cloud 10-Ks | 2.6 | **6.3** |
| Sourcegraph Handbook | 5.7 | 5.8 |
| Supreme Court Opinions | 6.1 | **7.4** |
| **Average** | **4.72** | **6.04** (+27.9%) |

CCH improves performance on every dataset. The biggest gains come from documents where chunks frequently lack explicit subject references (like financial reports where "the company" is used instead of the company name).

---
## Summary

| Step | What happened |
|---|---|
| 1 | Downloaded the Nike 2023 annual report |
| 2 | Split into 500 chunks (800 chars each) |
| 3 | Used the LLM to generate a descriptive document title |
| 4-5 | Compared relevance scores with/without the title header |

**Key insight:** CCH is one of the simplest RAG improvements — just prepend the document title to each chunk before embedding. This single change gives every chunk the context it needs to be matched correctly. Chunks that say "the company" instead of "Nike" suddenly become findable for Nike-related queries.